# 📊 月營收 YoY 回測分析｜單一 Cell 版

## 如何使用？

1. 把這份教材存到自己的 Colab（「**檔案**」→「**在雲端硬碟中儲存副本**」）
2. 修改下方程式碼開頭的「可調整參數」區塊
3. 點「**全部執行**」，等待結果

## 可調整的參數

| 參數 | 說明 | 範例 |
|------|------|------|
| `STOCK_ID` | 股票代號（不含後綴）| `"2330"` |
| `MARKET` | 上市填 `TW`，上櫃填 `TWO` | `"TW"` |
| `YOY_THRESHOLD` | YoY 觸發門檻（%） | `20` |
| `HOLD_DAYS` | 買入後持有天數 | `20` |
| `START_DATE` | 回測起始日期 | `"2015-01-01"` |
| `FINMIND_TOKEN` | FinMind Token（可留空）| `""` |

> ⚠️ 本教材僅供學術研究與程式教學用途，不構成任何投資建議。

---

## 如何用 AI 客製化這份程式？

把下方的程式碼複製，貼給 Claude 或 ChatGPT，說：

> 「我有一段月營收 YoY 回測的程式碼，請幫我改成 ＿＿＿＿」

例如：
- 「改成可以同時回測多支股票」
- 「把圖表改成互動式」
- 「加上停損條件，跌超過 5% 就出場」


In [ ]:
!pip install requests pandas yfinance matplotlib -q

import requests
import yfinance as yf
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import platform, warnings
warnings.filterwarnings("ignore")

# 字型設定
if platform.system() == "Windows":
    matplotlib.rc("font", family="Microsoft JhengHei")
matplotlib.rcParams["axes.unicode_minus"] = False

# ── 可調整參數 ────────────────────────────────────────
STOCK_ID      = "2330"        # 股票代號（不含後綴）
MARKET        = "TW"          # 上市填 TW，上櫃填 TWO
YOY_THRESHOLD = 20            # YoY 門檻（%），超過才觸發
HOLD_DAYS     = 20            # 持有交易日數
START_DATE    = "2015-01-01"  # 資料起始日
END_DATE      = "2025-12-31"  # 資料結束日
FINMIND_TOKEN = ""            # FinMind Token（選填）
WINDOWS       = [5, 10, 20, 30]  # 多時間窗口比較
# ─────────────────────────────────────────────────────

print(f"✅ 參數：{STOCK_ID}.{MARKET}，YoY>{YOY_THRESHOLD}%，持有 {HOLD_DAYS} 天")

# ── Step 1：抓月營收（FinMind）────────────────────────
params = {
    "dataset":    "TaiwanStockMonthRevenue",
    "data_id":    STOCK_ID,
    "start_date": START_DATE,
    "end_date":   END_DATE,
}
if FINMIND_TOKEN:
    params["token"] = FINMIND_TOKEN

resp = requests.get("https://api.finmindtrade.com/api/v4/data", params=params, timeout=30)
data = resp.json()

if data.get("status") != 200:
    print(f"❌ 抓取失敗：{data.get('msg')}")
else:
    rev = pd.DataFrame(data["data"])
    rev["date"] = pd.to_datetime(rev["date"])
    rev = rev.sort_values("date").reset_index(drop=True)
    rev["revenue_yoy"] = rev["revenue"].pct_change(12) * 100
    rev = rev.dropna(subset=["revenue_yoy"]).reset_index(drop=True)
    print(f"✅ 月營收：共 {len(rev)} 筆（{rev['date'].min().date()} ～ {rev['date'].max().date()}）")

    # ── Step 2：抓股價（yfinance），計算公告日 ────────────
    ticker = f"{STOCK_ID}.{MARKET}"
    price_df = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
    if isinstance(price_df.columns, pd.MultiIndex):
        price_df.columns = price_df.columns.get_level_values(0)
    price_df = price_df[["Close"]].copy()

    if price_df.empty:
        print(f"❌ 無法取得 {ticker} 股價，請確認 MARKET 設定是否正確")
    else:
        print(f"✅ 股價：共 {len(price_df)} 筆（{price_df.index.min().date()} ～ {price_df.index.max().date()}）")

        def get_announce_date(revenue_date):
            next_month = revenue_date + pd.DateOffset(months=1)
            announce   = pd.Timestamp(next_month.year, next_month.month, 10)
            future     = price_df.index[price_df.index >= announce]
            return future[0] if len(future) > 0 else None

        rev["announce_date"] = rev["date"].apply(get_announce_date)
        rev = rev.dropna(subset=["announce_date"]).reset_index(drop=True)

        # ── Step 3：篩選觸發點，計算勝率 ─────────────────────
        signals = rev[rev["revenue_yoy"] > YOY_THRESHOLD].copy()
        print(f"\nYoY > {YOY_THRESHOLD}% 的月份共 {len(signals)} 次")

        results = []
        for _, row in signals.iterrows():
            buy_idx  = price_df.index.get_loc(row["announce_date"])
            sell_idx = buy_idx + HOLD_DAYS
            if sell_idx < len(price_df):
                bp  = price_df["Close"].iloc[buy_idx]
                sp  = price_df["Close"].iloc[sell_idx]
                ret = (sp - bp) / bp * 100
                results.append({
                    "revenue_month": row["date"],
                    "announce_date": row["announce_date"],
                    "yoy":           row["revenue_yoy"],
                    "buy_price":     bp,
                    "sell_price":    sp,
                    "return_pct":    ret
                })

        result_df = pd.DataFrame(results)

        if result_df.empty:
            print("❌ 無有效樣本，請調整參數")
        else:
            win_rate = (result_df["return_pct"] > 0).mean() * 100
            avg_ret  = result_df["return_pct"].mean()
            print(f"\n{'='*45}")
            print(f"  股票：{STOCK_ID}　YoY 門檻：{YOY_THRESHOLD}%　持有：{HOLD_DAYS} 天")
            print(f"  樣本數：{len(result_df)}")
            print(f"  勝率：{win_rate:.1f}%")
            print(f"  平均報酬：{avg_ret:+.2f}%")
            print(f"{'='*45}")
            print("\n各年觸發次數：")
            print(result_df["revenue_month"].dt.year.value_counts().sort_index().to_string())

        # ── Step 4：多時間窗口比較 ────────────────────────────
        print(f"\n{'='*50}")
        print(f"  {STOCK_ID}　YoY > {YOY_THRESHOLD}%　多時間窗口勝率比較")
        print(f"{'='*50}")
        print(f"  {'持有天數':>8} {'勝率':>8} {'平均報酬':>10} {'樣本數':>8}")
        print(f"  {'-'*38}")
        for w in WINDOWS:
            w_results = []
            for _, row in signals.iterrows():
                buy_idx  = price_df.index.get_loc(row["announce_date"])
                sell_idx = buy_idx + w
                if sell_idx < len(price_df):
                    bp = price_df["Close"].iloc[buy_idx]
                    sp = price_df["Close"].iloc[sell_idx]
                    w_results.append((sp - bp) / bp * 100)
            if w_results:
                wr    = sum(1 for r in w_results if r > 0) / len(w_results) * 100
                ar    = sum(w_results) / len(w_results)
                emoji = "📈" if wr > 50 else "📉"
                print(f"  {w:>5} 天後  {emoji} {wr:>5.1f}%  {ar:>+8.2f}%  {len(w_results):>6}")
        print(f"{'='*50}")

        # ── Step 5：視覺化 ────────────────────────────────────
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=False)

        ax1.plot(price_df.index, price_df["Close"], color="#1f77b4", linewidth=1, label="收盤價")
        if not result_df.empty:
            ax1.scatter(
                result_df["announce_date"],
                result_df["buy_price"],
                color="red", marker="v", s=80, zorder=5,
                label=f"買入點（YoY>{YOY_THRESHOLD}%，共{len(result_df)}次）"
            )
        ax1.set_title(f"{STOCK_ID} 股價走勢與月營收 YoY 觸發點", fontsize=13)
        ax1.set_ylabel("股價（元）")
        ax1.legend()
        ax1.grid(alpha=0.3)

        colors = ["green" if v > 0 else "red" for v in rev["revenue_yoy"]]
        ax2.bar(rev["announce_date"], rev["revenue_yoy"], color=colors, width=20, alpha=0.7)
        ax2.axhline(YOY_THRESHOLD, color="orange", linewidth=1.5, linestyle="--",
                    label=f"門檻 {YOY_THRESHOLD}%")
        ax2.axhline(0, color="black", linewidth=0.8)
        ax2.set_title(f"{STOCK_ID} 月營收 YoY（%）", fontsize=13)
        ax2.set_ylabel("YoY (%)")
        ax2.legend()
        ax2.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()
        print("\n✅ 全部完成！")
